In [1]:
import requests
import io
import pandas as pd
import matplotlib.pyplot as plt
import re
import time

In [2]:
ADS_TOKEN = "TU_TOKEN_DE_ADS_AQUI"

def extraer_referencias_bibtex(df_columna, set_vistos):
    bibtex_entries = []
    for item in df_columna.dropna():
        # Se corrigió la regex para admitir los puntos (.) y signos (&) del bibcode
        match = re.search(r'refstr=([A-Za-z0-9\.\-\&]+).*?href=(.*?)(?:\s|>|$)', str(item))
        if match:
            refstr = match.group(1)
            url = match.group(2).strip('"\'')
            bibcode = refstr # En Exoplanet Archive, el refstr ES el bibcode de ADS
            
            if refstr not in set_vistos:
                set_vistos.add(refstr)
                bibtex_obtenido = False
                
                # Intentar obtener el BibTeX real desde la API de ADS
                if ADS_TOKEN and ADS_TOKEN != "TU_TOKEN_DE_ADS_AQUI":
                    headers = {
                        "Authorization": f"Bearer {ADS_TOKEN}",
                        "Content-Type": "application/json"
                    }
                    url_api = "https://api.adsabs.harvard.edu/v1/export/bibtex"
                    
                    try:
                        resp = requests.post(url_api, headers=headers, json={"bibcode": [bibcode]})
                        if resp.status_code == 200:
                            data = resp.json()
                            bibtex_real = data.get('export', '').strip()
                            if bibtex_real:
                                bibtex_entries.append(bibtex_real)
                                bibtex_obtenido = True
                                time.sleep(0.3) # Evitar bloqueo por exceso de peticiones
                    except Exception as e:
                        print(f"Error consultando ADS para {bibcode}: {e}")
                
                # Fallback: Si no hay token o la API falla, usar un formato básico
                if not bibtex_obtenido:
                    año = bibcode[:4] if bibcode[:4].isdigit() else "20XX"
                    bib_entry = f"""@article{{{refstr},
    author = {{{refstr.replace('_', ' ').title()}}},
    title = {{Parámetros extraídos de Exoplanet Archive ({bibcode})}},
    journal = {{NASA Exoplanet Archive References}},
    url = {{{url}}},
    year = {{{año}}}
}}"""
                    bibtex_entries.append(bib_entry)
                    
    return bibtex_entries

In [3]:
def mineria_sistemas_exoplanetarios(tipo_espectral, masa_min, masa_max):
    url_base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    
    # Actualizado con alias en inglés (en minúsculas por requerimiento de la API)
    query = f"""
    SELECT 
        hostname AS host_name, 
        sy_pnum AS number_of_planets, 
        sy_dist AS distance_pc, sy_disterr1 AS distance_pc_err1, sy_disterr2 AS distance_pc_err2,
        st_spectype AS spectral_type, 
        st_teff AS temp_k, st_tefferr1 AS temp_k_err1, st_tefferr2 AS temp_k_err2,
        st_rad AS stellar_radius_sol, st_raderr1 AS stellar_radius_sol_err1, st_raderr2 AS stellar_radius_sol_err2,
        st_mass AS stellar_mass_sol, st_masserr1 AS stellar_mass_sol_err1, st_masserr2 AS stellar_mass_sol_err2,
        st_teff_reflink AS stellar_ref,
        pl_name AS planet_name, 
        pl_orbper AS orbital_period_days, pl_orbpererr1 AS orbital_period_days_err1, pl_orbpererr2 AS orbital_period_days_err2,
        pl_orbsmax AS semi_major_axis_au, pl_orbsmaxerr1 AS semi_major_axis_au_err1, pl_orbsmaxerr2 AS semi_major_axis_au_err2,
        pl_rade AS planet_radius_earth, pl_radeerr1 AS planet_radius_earth_err1, pl_radeerr2 AS planet_radius_earth_err2,
        pl_bmasse AS planet_mass_earth, pl_bmasseerr1 AS planet_mass_earth_err1, pl_bmasseerr2 AS planet_mass_earth_err2,
        pl_bmassj AS planet_mass_jup, pl_bmassjerr1 AS planet_mass_jup_err1, pl_bmassjerr2 AS planet_mass_jup_err2,
        pl_msinie AS mpsini_earth, pl_msinieerr1 AS mpsini_earth_err1, pl_msinieerr2 AS mpsini_earth_err2,
        pl_msinij AS mpsini_jup, pl_msinijerr1 AS mpsini_jup_err1, pl_msinijerr2 AS mpsini_jup_err2,
        discoverymethod AS discovery_method, 
        pl_orbper_reflink AS planet_ref
    FROM pscomppars
    WHERE sy_pnum BETWEEN 3 AND 8
      AND st_spectype LIKE '%{tipo_espectral}%'
      AND (
          (pl_bmassj >= {masa_min} AND pl_bmassj <= {masa_max}) OR 
          (pl_msinij >= {masa_min} AND pl_msinij <= {masa_max})
      )
    ORDER BY sy_dist ASC
    """
    
    respuesta = requests.get(url_base, params={"query": query, "format": "csv"})
    if respuesta.status_code != 200: return None, None
    
    df = pd.read_csv(io.StringIO(respuesta.text))
    if df.empty: return None, None
    
    # 1. TABLA ESTRELLA
    cols_estrellas_lower = [
        'host_name', 'number_of_planets', 'distance_pc', 'distance_pc_err1', 'distance_pc_err2',
        'spectral_type', 'temp_k', 'temp_k_err1', 'temp_k_err2',
        'stellar_radius_sol', 'stellar_radius_sol_err1', 'stellar_radius_sol_err2',
        'stellar_mass_sol', 'stellar_mass_sol_err1', 'stellar_mass_sol_err2', 'stellar_ref'
    ]
    df_star = df[cols_estrellas_lower].copy()
    
    # Asignar nombres formales con mayúsculas
    df_star.columns = [
        'Host_Name', 'Number_of_Planets', 'Distance_pc', 'Distance_pc_err1', 'Distance_pc_err2',
        'Spectral_Type', 'Temp_K', 'Temp_K_err1', 'Temp_K_err2',
        'Stellar_Radius_Sol', 'Stellar_Radius_Sol_err1', 'Stellar_Radius_Sol_err2',
        'Stellar_Mass_Sol', 'Stellar_Mass_Sol_err1', 'Stellar_Mass_Sol_err2', 'Stellar_Ref'
    ]
                       
    df_star['Ref_Year'] = df_star['Stellar_Ref'].str.extract(r'(\d{4})').astype(float)
    df_star = df_star.sort_values(by=['Host_Name', 'Ref_Year'], ascending=[True, False]).drop_duplicates(subset=['Host_Name'], keep='first')
    df_star = df_star.drop(columns=['Ref_Year']).reset_index(drop=True)
    
    # 2. TABLA PLANETAS
    cols_planetas_lower = [
        'planet_name', 'orbital_period_days', 'orbital_period_days_err1', 'orbital_period_days_err2',
        'semi_major_axis_au', 'semi_major_axis_au_err1', 'semi_major_axis_au_err2',
        'planet_mass_jup', 'planet_mass_jup_err1', 'planet_mass_jup_err2',
        'mpsini_jup', 'mpsini_jup_err1', 'mpsini_jup_err2',
        'planet_mass_earth', 'planet_mass_earth_err1', 'planet_mass_earth_err2',
        'mpsini_earth', 'mpsini_earth_err1', 'mpsini_earth_err2',
        'planet_radius_earth', 'planet_radius_earth_err1', 'planet_radius_earth_err2',
        'discovery_method', 'planet_ref'
    ]
    df_planets = df[cols_planetas_lower].copy()
    
    # Asignar nombres formales con mayúsculas
    df_planets.columns = [
        'Planet_Name', 'Orbital_Period_Days', 'Orbital_Period_Days_err1', 'Orbital_Period_Days_err2',
        'Semi_Major_Axis_AU', 'Semi_Major_Axis_AU_err1', 'Semi_Major_Axis_AU_err2',
        'Planet_Mass_Jup', 'Planet_Mass_Jup_err1', 'Planet_Mass_Jup_err2',
        'Mpsini_Jup', 'Mpsini_Jup_err1', 'Mpsini_Jup_err2',
        'Planet_Mass_Earth', 'Planet_Mass_Earth_err1', 'Planet_Mass_Earth_err2',
        'Mpsini_Earth', 'Mpsini_Earth_err1', 'Mpsini_Earth_err2',
        'Planet_Radius_Earth', 'Planet_Radius_Earth_err1', 'Planet_Radius_Earth_err2',
        'Discovery_Method', 'Planet_Ref'
    ]
    
    return df_star, df_planets

In [4]:
def combinar_errores(df, cols, is_latex=False):
    df_out = df.copy()
    for col in cols:
        if col in df_out.columns:
            err1_col, err2_col = f"{col}_err1", f"{col}_err2"
            if err1_col in df_out.columns and err2_col in df_out.columns:
                def fmt(row):
                    val = row[col]
                    if pd.isna(val): return "--"
                    
                    e1 = row[err1_col]
                    e2 = row[err2_col]
                    
                    if isinstance(val, float): val = round(val, 4)
                    if isinstance(e1, float): e1 = round(e1, 4)
                    if isinstance(e2, float): e2 = round(e2, 4)
                    
                    if pd.isna(e1) or pd.isna(e2): return str(val)
                        
                    if is_latex:
                        if abs(e1 + e2) < 1e-7: 
                            return f"${val} \\pm {e1}$"
                        return f"${val}^{{+{e1}}}_{{{e2}}}$"
                    else:
                        if abs(e1 + e2) < 1e-7: 
                            return f"{val} ± {e1}"
                        return f"{val} +{e1} {e2}"
                
                df_out[col] = df_out.apply(fmt, axis=1)
                df_out = df_out.drop(columns=[err1_col, err2_col])
    return df_out

In [5]:
def convertir_a_cite(texto):
    if pd.isna(texto): return "--"
    match = re.search(r'refstr=([\w_]+)', str(texto))
    if match: return f"\\cite{{{match.group(1)}}}"
    return str(texto)


def generar_reporte(df_star, df_planets, filtro_espectral, masa_min, masa_max):
    nombre_base = f"Sistemas_{filtro_espectral}_{masa_min}a{masa_max}MJ"
    
    cols_con_errores = [
        'Distance_pc', 'Temp_K', 'Stellar_Radius_Sol', 'Stellar_Mass_Sol', 
        'Orbital_Period_Days', 'Semi_Major_Axis_AU', 'Planet_Mass_Jup', 'Mpsini_Jup', 
        'Planet_Mass_Earth', 'Mpsini_Earth', 'Planet_Radius_Earth'
    ]
                        
    # 1. Archivos CSV
    df_star_csv = combinar_errores(df_star, cols_con_errores, is_latex=False)
    df_planets_csv = combinar_errores(df_planets, cols_con_errores, is_latex=False)
    df_star_csv.to_csv(f"Tabla_Estrellas_{nombre_base}.csv", index=False, encoding='utf-8')
    df_planets_csv.to_csv(f"Tabla_Planetas_{nombre_base}.csv", index=False, encoding='utf-8')
    
    # 2. Generación BibTeX
    print("Obteniendo referencias de la API de ADS... esto puede tomar unos segundos.")
    set_vistos = set()
    bib_estrellas = extraer_referencias_bibtex(df_star['Stellar_Ref'], set_vistos)
    bib_planetas = extraer_referencias_bibtex(df_planets['Planet_Ref'], set_vistos)
    with open(f"Bibliografias_{nombre_base}.bib", "w", encoding='utf-8') as f:
        f.write("\n\n".join(bib_estrellas + bib_planetas))
        
    # 3. Exportación LaTeX
    df_star_latex = combinar_errores(df_star, cols_con_errores, is_latex=True)
    df_planets_latex = combinar_errores(df_planets, cols_con_errores, is_latex=True)
    
    df_star_latex['Stellar_Ref'] = df_star_latex['Stellar_Ref'].apply(convertir_a_cite)
    df_planets_latex['Planet_Ref'] = df_planets_latex['Planet_Ref'].apply(convertir_a_cite)
    
    df_star_latex.columns = [c.replace('_', ' ') for c in df_star_latex.columns]
    df_planets_latex.columns = [c.replace('_', ' ') for c in df_planets_latex.columns]
    
    archivo_tex_estrellas = f"Tabla_Estrellas_{filtro_espectral}.tex"
    archivo_tex_planetas = f"Tabla_Planetas_{filtro_espectral}_Planetas.tex"
    
    # Manejar compatibilidad de versiones de Pandas para ocultar el índice
    with open(archivo_tex_estrellas, "w", encoding='utf-8') as f:
        try:
            latex_str = df_star_latex.style.format(na_rep='--').hide(axis="index").to_latex(hrules=True)
        except AttributeError: # Fallback para pandas < 1.4
            latex_str = df_star_latex.style.format(na_rep='--').hide_index().to_latex(hrules=True)
        f.write(latex_str)
        
    with open(archivo_tex_planetas, "w", encoding='utf-8') as f:
        try:
            latex_str = df_planets_latex.style.format(na_rep='--').hide(axis="index").to_latex(hrules=True)
        except AttributeError: # Fallback para pandas < 1.4
            latex_str = df_planets_latex.style.format(na_rep='--').hide_index().to_latex(hrules=True)
        f.write(latex_str)
        
    print("✅ Reportes con análisis de errores y formato LaTeX generados con éxito.")
    print(f"📄 Archivos LaTeX creados: {archivo_tex_estrellas} y {archivo_tex_planetas}")
    print(f"📚 Archivo de bibliografía creado: Bibliografias_{nombre_base}.bib")

In [9]:
if __name__ == "__main__":
    espectro = "G"
    m_min, m_max = 0.0001, 2.0
    
    star, planets = mineria_sistemas_exoplanetarios(espectro, m_min, m_max)
    
    if star is not None and planets is not None:
        generar_reporte(star, planets, espectro, m_min, m_max)
    else:
        print("No se encontraron resultados.")

Obteniendo referencias de la API de ADS... esto puede tomar unos segundos.
✅ Reportes con análisis de errores y formato LaTeX generados con éxito.
📄 Archivos LaTeX creados: Tabla_Estrellas_G.tex y Tabla_Planetas_G_Planetas.tex
📚 Archivo de bibliografía creado: Bibliografias_Sistemas_G_0.0001a2.0MJ.bib
